# Training at Scale


This lecture is about the engineering that sits between "here is a transformer
and a loss" and "here is a trained model": what training costs, what a GPU can
actually do, how a training step uses memory, the tricks that fit a bigger model
on one GPU, the ways to spread it over many, and the empirical laws that say how
to spend a compute budget. UDL does not cover this material; the notation
follows the primary papers, and each is cited where it is used. The two numbers
to carry out of this lecture are $C \approx 6ND$ FLOPs for training and about
16 bytes per parameter of optimizer state. You will use both in the midterm
project on the SCC.


## Today's roadmap

1. Why this matters: what training a transformer costs, and the $6ND$ rule
2. GPU mechanics: FLOPs vs. bandwidth, the roofline, memory accounting
3. Fitting more on one GPU: mixed precision, gradient accumulation, checkpointing, FlashAttention
4. Parallelism: data, ZeRO/FSDP, tensor, pipeline, expert
5. Scaling laws: Kaplan, Chinchilla, and what came after
6. Data at scale: curation, deduplication, filtering, mixing
7. Instrumentation: loss curves, throughput, reading an experiment log
8. Summary


# Why this matters

Lecture 11 looked inside a real open-weight transformer. Before we can train
one, even a small one, we need to know what it will cost and whether it fits.
This section gives the one-line cost estimate that every later section builds on.


## What a frontier training run costs

| Model | Params | Tokens | Hardware | GPU-hours | Source |
|---|---|---|---|---|---|
| Llama 3 405B (2024) | 405B dense | 15.6T | up to 16,384 H100 | 30.84M | [Grattafiori et al. 2024](https://arxiv.org/abs/2407.21783), [model card](https://github.com/meta-llama/llama-models/blob/main/models/llama3_1/MODEL_CARD.md) |
| DeepSeek-V3 (2024) | 671B total, 37B active (MoE) | 14.8T | 2,048 H800 | 2.788M | [DeepSeek-AI 2024](https://arxiv.org/abs/2412.19437) |

- At the \$2 per GPU-hour that the DeepSeek-V3 paper assumes, that is roughly
  \$60M of compute for Llama 3 405B and \$5.6M for DeepSeek-V3.
- The 11× gap comes from choices this lecture covers: a mixture-of-experts
  (37B active parameters), FP8 training, and a parallelism layout designed
  around the interconnect.
- Over a 54-day stretch of the Llama 3 run, there were 466 job interruptions,
  419 of them unplanned, and about 78% of those were hardware
  ([Grattafiori et al. 2024](https://arxiv.org/abs/2407.21783), section 3.3.4).
  At scale, training is also a reliability problem.


## Where the FLOPs go: the $6ND$ rule

A dense transformer with $N$ parameters, trained on $D$ tokens, costs about

$$
C \approx 6\,N\,D \quad \text{floating-point operations.}
$$

- **Forward, $2N$ per token.** Almost every parameter sits in a matrix
  multiply, and each parameter does one multiply and one add per token.
- **Backward, $4N$ per token.** Each matmul $\mathbf{Y} = \mathbf{X}\mathbf{W}$
  needs two matmuls on the way back: $\partial L/\partial \mathbf{X}$ (to keep
  going) and $\partial L/\partial \mathbf{W}$ (the gradient). Twice the
  forward cost.
- **Optimizer, negligible.** Adam touches each parameter once per *step*,
  not once per *token*; with thousands of tokens per step it disappears.
- Attention adds $2\,n_\text{layer}\,n_\text{ctx}\,d_\text{model}$ per token
  in the forward pass, small unless the context is long.


Source: [Kaplan et al. 2020](https://arxiv.org/abs/2001.08361), section 2.1;
the same accounting is used to define MFU in
[Chowdhery et al. 2022](https://arxiv.org/abs/2204.02311), appendix B.

## Worked calculation: a GPT-2-small-sized run

Train $N = 124\text{M}$ parameters on $D = 10\text{B}$ tokens.

1. Compute: $C = 6 \times 1.24\times10^{8} \times 10^{10} = 7.4 \times 10^{18}$ FLOPs.
2. An A100 peaks at 312 TFLOP/s in bf16; a well-tuned run reaches about 40% of
   that, so $1.25\times10^{14}$ FLOP/s.
3. Time: $7.4\times10^{18} / 1.25\times10^{14} = 6.0\times10^{4}$ s, about
   **16.5 hours** on one A100.


**Sanity check at the other end.** Llama 3 405B:
$6 \times 4.05\times10^{11} \times 1.56\times10^{13} = 3.8\times10^{25}$ FLOPs,
the number the paper reports. At the 400 TFLOP/s per H100 it achieved, that is
$9.5\times10^{10}$ GPU-seconds, about 26M GPU-hours; the model card's 30.84M
includes restarts and the long-context and annealing stages.


The rule is quizzable: given any two of $N$, $D$, and GPU-hours, you can
estimate the third.


Both calculations as a calculator. Change the model, the token count, the GPU
or the MFU and rerun.

In [ ]:
A100_BF16 = 312e12   # dense bf16 tensor-core peak, FLOP/s
H100_BF16 = 989e12

def training_flops(N, D):
    return 6 * N * D

def training_seconds(N, D, peak_flops, mfu=0.40, n_gpus=1):
    return training_flops(N, D) / (peak_flops * mfu * n_gpus)

# GPT-2-small-sized run on one A100 at 40% MFU
N, D, mfu = 124e6, 10e9, 0.40
C = training_flops(N, D)
sec = training_seconds(N, D, A100_BF16, mfu)
print(f"N = 124M, D = 10B:  C = {C:.2e} FLOPs,  {sec:.2e} s = {sec / 3600:.1f} hours on one A100")

# Llama 3 405B at the 400 TFLOP/s per H100 it achieved
N, D, achieved = 405e9, 15.6e12, 400e12
C = training_flops(N, D)
gpu_seconds = C / achieved
print(f"Llama 3 405B:       C = {C:.1e} FLOPs,  {gpu_seconds:.1e} GPU-s = {gpu_seconds / 3600 / 1e6:.1f}M H100-hours")
print(f"  on 16,384 H100s: {gpu_seconds / 16_384 / 86_400:.0f} days;"
      f"  on one A100 at 40% MFU: {training_seconds(N, D, A100_BF16) / 86_400 / 365:,.0f} years")

# Any two of N, D and GPU-hours give the third: tokens a 124M model gets in 100 A100-hours
N, gpu_hours = 124e6, 100
D = gpu_hours * 3600 * A100_BF16 * 0.40 / (6 * N)
print(f"100 A100-hours at 40% MFU train a 124M model on {D:.1e} tokens")

## What one SCC GPU can and cannot do

![Chinchilla-optimal model size reachable with a given budget, assuming 40% MFU and the $6ND$ rule with $D = 20N$ (section 5). Peak FLOP/s from NVIDIA datasheets. Source: instructor figure.](figs/12-Training-at-Scale/gpu-budget.png)

- One A100-day is about $10^{19}$ FLOPs: a compute-optimal model of roughly
  300M parameters on 6B tokens. A week buys about 800M parameters.
- Llama 3 405B would take one A100 about 10,000 years. Nobody trains
  frontier models on one GPU; everybody *develops* on one.
- The SCC's GPU nodes include A100s (40 and 80 GB) and L40S cards; the
  midterm project runs on one of them.


# GPU mechanics

The $6ND$ rule counts arithmetic. Whether a GPU actually delivers its peak
arithmetic rate depends on whether it can feed the arithmetic units with data
fast enough, which is the subject of this section.


## A GPU in three numbers

| | A100 80GB | H100 SXM |
|---|---|---|
| bf16 tensor-core peak (dense) | 312 TFLOP/s | 989 TFLOP/s |
| HBM bandwidth | 2.0 TB/s | 3.35 TB/s |
| HBM capacity | 80 GB | 80 GB |
| NVLink (GPU to GPU, in a node) | 600 GB/s | 900 GB/s |
| Power | 400 W | 700 W |

Source: NVIDIA [A100](https://www.nvidia.com/en-us/data-center/a100/) and
[H100](https://www.nvidia.com/en-us/data-center/h100/) datasheets.

- Compute grew 3.2× between generations; bandwidth grew 1.7×. The gap between
  "how fast it can multiply" and "how fast it can fetch" keeps widening.
- Memory *capacity* did not grow at all. Fitting the model is a separate
  problem from running it fast.
- The GPU also has about 20 MB of on-chip SRAM at roughly 19 TB/s
  ([Dao et al. 2022](https://arxiv.org/abs/2205.14135)): tiny, but ten times
  faster than HBM. FlashAttention lives there.


## Arithmetic intensity and the roofline

For a kernel, define

$$
I = \frac{\text{FLOPs performed}}{\text{bytes moved to and from HBM}} \quad (\text{FLOP/byte}).
$$

A kernel cannot run faster than either limit:

$$
\text{throughput} \le \min\big(\text{peak FLOP/s},\; I \times \text{bandwidth}\big).
$$

- The crossover, $I^\ast = \text{peak}/\text{bandwidth}$, is the **ridge
  point**: about 156 FLOP/byte on an A100 80GB, about 295 on an H100.
- Below the ridge the kernel is **memory-bound**: the arithmetic units wait for
  data. Above it, **compute-bound**: the data arrives faster than it can be
  used. Only compute-bound kernels see the datasheet number.


Source: [Williams, Waterman and Patterson 2009](https://doi.org/10.1145/1498765.1498785).
A readable deep-learning treatment is
[He 2022, *Making Deep Learning Go Brrrr*](https://horace.io/brrr_intro.html).

## The roofline picture

In [ ]:
#| code-fold: true
#| out-width: "78%"
#| fig-cap: "Roofline for an A100 80GB and an H100 SXM in bf16, with three kernels placed on the A100 line. Computed from datasheet numbers."
#| fig-alt: "Log-log roofline plot: attainable TFLOP/s rises linearly with arithmetic intensity until it hits the peak. An elementwise add at 0.17 FLOP per byte reaches 0.3 TFLOP/s; a 16-token GEMM at 16 FLOP per byte reaches about 30 TFLOP/s; a 4096-token GEMM at about 1,400 FLOP per byte sits on the 312 TFLOP/s roof. Ridge points at 156 and 295 FLOP per byte are marked."
import numpy as np
import matplotlib.pyplot as plt

gpus = {"A100 80GB": (312, 2.0), "H100 SXM": (989, 3.35)}   # bf16 TFLOP/s, HBM TB/s
kernels = {                                                  # FLOP/byte, placed on the A100 line
    "elementwise add\n(1 FLOP / 6 bytes)": 1 / 6,
    "GEMM, 16 tokens\n(decode)": 16,
    "GEMM, 4096 tokens\n(d = 4096)": 4096 * 4096 / (2 * 4096 + 4096),
}

def attainable(intensity, peak, bw):
    return np.minimum(peak, bw * intensity)   # TB/s x FLOP/byte = TFLOP/s

fig, ax = plt.subplots(figsize=(8.5, 4.4))
x = np.logspace(-1.2, 4, 400)
for (name, (peak, bw)), color in zip(gpus.items(), ["C0", "C1"]):
    ax.loglog(x, attainable(x, peak, bw), color=color, lw=2,
              label=f"{name}: {peak} TFLOP/s, {bw} TB/s, ridge {peak / bw:.0f} FLOP/B")
    ax.axvline(peak / bw, color=color, lw=0.8, ls=":")
peak, bw = gpus["A100 80GB"]
for label, I in kernels.items():
    y = attainable(I, peak, bw)
    ax.plot(I, y, "ko", ms=6)
    ax.annotate(f"{label}: {y:.3g} TFLOP/s", (I, y), textcoords="offset points",
                xytext=(8, -32 if I > 100 else 8), fontsize=9)
ax.set(xlim=(0.08, 1e4), ylim=(0.1, 2000),
       xlabel="arithmetic intensity (FLOP per byte moved from HBM)",
       ylabel="attainable throughput (TFLOP/s)")
ax.legend(loc="lower right", fontsize=9, frameon=False)
plt.show()

## Why batch size and matrix shapes matter

A bf16 matmul $\mathbf{X}\mathbf{W}$ with $\mathbf{X} \in \mathbb{R}^{B\times d}$
and $\mathbf{W} \in \mathbb{R}^{d\times d}$ does $2Bd^2$ FLOPs and moves
$2(Bd + d^2 + Bd)$ bytes, so

$$
I = \frac{B\,d}{2B + d} \;\approx\; \begin{cases} B & B \ll d \\ d/2 & B \gg d. \end{cases}
$$

![Arithmetic intensity of a bf16 GEMM against the number of token rows. Source: instructor figure.](figs/12-Training-at-Scale/gemm-intensity.png)

- With few tokens the intensity *is* the token count: every weight is read from
  HBM once and used $B$ times. Decoding one token at a time ($B = 1$) is
  hopelessly memory-bound; this returns in lecture 15.
- Training is compute-bound as long as each matmul sees a few hundred tokens,
  which a micro-batch of even one 2048-token sequence provides.
- Dimensions that are multiples of 64 or 128 fill the tensor cores; a
  vocabulary of 50,257 is slower than one of 50,304.


Where the crossover happens for your own matrix shapes:

In [ ]:
ridges = {"A100": 312e12 / 2.0e12, "H100": 989e12 / 3.35e12}   # FLOP/byte

def gemm_intensity(B, d, bytes_per_value=2):
    flops = 2 * B * d * d
    bytes_moved = bytes_per_value * (B * d + d * d + B * d)   # read X and W, write Y
    return flops / bytes_moved

d = 4096
for B in [1, 16, 128, 512, 2048, 16_384]:
    I = gemm_intensity(B, d)
    regime = "  ".join(f"{g}: {'compute' if I > r else 'memory'}-bound" for g, r in ridges.items())
    print(f"B = {B:>6}, d = {d}:  I = {I:7.1f} FLOP/byte   {regime}")
for g, r in ridges.items():   # solve B d / (2B + d) = r for B
    print(f"{g}: compute-bound once B > {r * d / (d - 2 * r):.0f} tokens per matmul (d = {d})")

## Memory accounting for a training step

With Adam and bf16 mixed precision, each parameter costs

| State | Precision | Bytes / param |
|---|---|---|
| weights (used in forward and backward) | bf16 | 2 |
| gradients | bf16 | 2 |
| master copy of the weights | fp32 | 4 |
| Adam first moment $\mathbf{m}$ | fp32 | 4 |
| Adam second moment $\mathbf{v}$ | fp32 | 4 |
| **total model states** | | **16** |

Source: [Rajbhandari et al. 2020](https://arxiv.org/abs/1910.02054), section 3.

- Plain fp32 training is *also* 16 bytes per parameter (4 + 4 + 8). Mixed
  precision does not shrink the model states; it shrinks activations and
  doubles the arithmetic rate.
- A 7B model needs 112 GB of states before the first activation is stored. It
  does not fit on an 80 GB GPU without sharding (section 4).


## Activations: the term that depends on batch size

Per transformer layer, for sequence length $s$, micro-batch $b$, hidden size
$h$, $a$ heads, bf16, the activations saved for the backward pass take about

$$
\text{bytes per layer} \approx s\,b\,h\,\Big(34 + 5\,\frac{a\,s}{h}\Big).
$$

Source: [Korthikanti et al. 2022](https://arxiv.org/abs/2205.05198), eq. 2
(attention block $11sbh + 5as^2b$, MLP $19sbh$, two layer norms $4sbh$).

- The $34\,sbh$ part is linear in tokens: inputs to each matmul, the GeLU
  input, dropout masks.
- The $5\,a s^2 b$ part is the attention score matrix and its softmax, stored
  for every head. It grows with $s^2$ and dominates at long context.
  FlashAttention removes it.
- Activation checkpointing (next section) replaces the whole thing with
  roughly $2\,sbh$ per layer plus one layer's worth.


## Worked example: a 1B model on one GPU

Take $L = 18$ layers, $h = 2048$, $a = 16$, vocabulary 50k:
$N \approx 12Lh^2 + Vh = 1.0\text{B}$. Micro-batch $b = 8$ sequences of $s = 2048$.

![Memory for one training step of the example model. Source: instructor figure, computed from the formulas on the previous two slides.](figs/12-Training-at-Scale/memory-breakdown.png)

- States: $16 \times 1.0\times10^{9} = 16$ GB.
- Activations, standard attention: $18 \times (34 + 5\cdot16\cdot2048/2048)\,sbh = 18 \times 114 \times 33.5\text{M} = 69$ GB. Total 85 GB: **does not fit** an 80 GB A100.
- With FlashAttention the $5as^2b$ term goes away: $18 \times 34 \times 33.5\text{M} = 20$ GB. Total 37 GB: fits an 80 GB card, not a 40 GB one.
- With checkpointing as well: about 2 GB of activations. Total 18 GB: fits anywhere, at about a third more compute.
- Not in the chart: the fp32 logits ($s\,b\,V \times 4$ bytes $= 3.3$ GB here), CUDA context, allocator fragmentation. Budget 10 to 20% headroom.


The accounting behind the chart. Change the shape, the micro-batch or the
sequence length and see which configurations fit.

In [ ]:
L, h, a, V = 18, 2048, 16, 50_000   # layers, hidden size, heads, vocabulary
s, b = 2048, 8                       # sequence length, micro-batch (sequences)
GB = 1e9

N = 12 * L * h**2 + V * h
states = 16 * N                      # bf16 weights and grads, fp32 master, Adam m and v
sbh = s * b * h
activations = {                      # bytes saved for backward (Korthikanti et al. 2022)
    "standard attention": L * sbh * (34 + 5 * a * s / h),
    "FlashAttention": L * sbh * 34,
    "FlashAttention + checkpointing": L * 2 * sbh + 34 * sbh,   # layer inputs + one live layer
}

print(f"N = {N / 1e9:.2f}B parameters, model states = {states / GB:.1f} GB")
for name, act in activations.items():
    total = (states + act) / GB
    print(f"{name:<31} activations {act / GB:5.1f} GB, total {total:5.1f} GB"
          f"   fits 40 GB: {total < 40!s:<5}  fits 80 GB: {total < 80}")
print(f"not counted: fp32 logits s*b*V*4 = {s * b * V * 4 / GB:.1f} GB")
print(f"a 7B model's states alone: {16 * 7e9 / GB:.0f} GB")

# Fitting more on one GPU

The accounting above says a 1B model is tight on one GPU and a 7B model is
impossible. Four techniques change those numbers, and every one of them trades
something (precision, step latency, recomputation, kernel complexity) for memory
or speed.


## Number formats

![Bit layouts of the five formats used in training. Maxima from the IEEE 754 and FP8 definitions. Source: instructor figure.](figs/12-Training-at-Scale/number-formats.png)

- **Exponent bits set the range.** fp16 overflows at 65,504 and flushes
  gradients below about $6\times10^{-8}$ to zero. bf16 keeps fp32's eight
  exponent bits, so it keeps fp32's range.
- **Mantissa bits set the precision.** bf16 has 7, about three decimal digits.
  That is enough for weights and activations in a matmul, not for accumulating
  a sum of thousands of products or for a running average of updates.
- **fp8** comes in two flavours: E4M3 for weights and activations (precision),
  E5M2 for gradients (range)
  ([Micikevicius et al. 2022](https://arxiv.org/abs/2209.05433)).


## The mixed-precision recipe

1. Keep an **fp32 master copy** of the weights; the optimizer updates it.
2. Cast weights to bf16 (or fp16) for the forward and backward passes; matmuls
   run on the tensor cores at 16-bit rates, accumulating in fp32 internally.
3. Gradients come out in 16 bits; the update $\boldsymbol{\phi} \leftarrow \boldsymbol{\phi} - \alpha\,\mathbf{g}$ is applied in fp32, where a small update to a large weight is not rounded away.
4. **Loss scaling (fp16 only).** Multiply the loss by $S$ (say $2^{16}$) before
   backward so small gradients do not underflow, divide the gradients by $S$
   before the update, and halve $S$ when an overflow appears. bf16 has the
   range and does not need it.


Sources: [Micikevicius et al. 2018](https://arxiv.org/abs/1710.03740) (fp16
recipe), [Kalamkar et al. 2019](https://arxiv.org/abs/1905.12322) (bf16).

```python
with torch.autocast("cuda", dtype=torch.bfloat16):
    loss = model(x, y)          # matmuls in bf16, reductions in fp32
loss.backward()                 # fp32 master weights live in the optimizer
optimizer.step()
```

## FP8, and where low precision breaks

DeepSeek-V3 was pre-trained with most matmuls in FP8
([DeepSeek-AI 2024](https://arxiv.org/abs/2412.19437), section 3.3):

- **Fine-grained scaling.** One scale per $1\times128$ tile of activations and
  per $128\times128$ block of weights, instead of one per tensor, so a single
  outlier does not force the whole tensor to lose precision.
- **High-precision accumulation.** Partial sums are promoted to fp32 registers
  every 128 elements, because the tensor cores' internal accumulator was not
  precise enough.
- **Kept in bf16 or fp32:** the embedding, the output head, the normalization
  layers, attention softmax, and all optimizer states. Everything that is a
  *reduction* or that is sensitive to a few large values.


The pattern generalizes: matmuls tolerate low precision; sums, averages,
exponentials and anything with outliers do not. When a run in low precision
diverges, these are the places to look.


## Gradient accumulation

The *global batch* (tokens per optimizer step) is a modelling choice, usually
0.5M to 16M tokens for LLMs. The *micro-batch* (tokens per forward pass) is a
memory choice. They need not be equal:

```python
for i, (x, y) in enumerate(loader):
    with torch.autocast("cuda", dtype=torch.bfloat16):
        loss = model(x, y) / accum_steps     # keep the mean over the global batch
    loss.backward()                           # gradients add up in .grad
    if (i + 1) % accum_steps == 0:
        optimizer.step(); optimizer.zero_grad()
```

- Memory scales with the micro-batch; the optimizer sees the global batch.
  Same gradient, same model, more wall-clock time per step.
- The one thing it does not fix: a micro-batch too small to make the
  matmuls compute-bound (previous section). The $6ND$ FLOPs still happen, just
  less efficiently.
- With data parallelism the all-reduce can be skipped on the non-final
  micro-steps (`no_sync()` in PyTorch).


## Activation checkpointing

![Store every layer's activations, or store every fourth layer's and recompute the rest during the backward pass. Source: instructor figure.](figs/12-Training-at-Scale/activation-checkpointing.png)

- Keep activations at $\sqrt{L}$ checkpoints; when backward reaches a segment,
  rerun its forward from the last checkpoint. Memory $O(\sqrt{L})$ instead of
  $O(L)$, for one extra forward pass
  ([Chen et al. 2016](https://arxiv.org/abs/1604.06174)).
- Cost: the forward is a third of $6ND$, so full recomputation adds about 33%
  compute. Megatron's *selective* recomputation recomputes only the cheap,
  memory-hungry attention pieces and costs a few percent
  ([Korthikanti et al. 2022](https://arxiv.org/abs/2205.05198)).
- Trade: compute you have for memory you do not. Prefer a larger micro-batch
  with checkpointing over a tiny one without.


## FlashAttention: an I/O-aware idea

![Left: the memory hierarchy, with SRAM an order of magnitude faster than HBM. Centre: blocks of $\mathbf{K}$, $\mathbf{V}$ and $\mathbf{Q}$ are streamed into SRAM and the softmax is computed incrementally. Right: attention time on GPT-2. Figure 1 of Dao et al., 2022.](figs/12-Training-at-Scale/flashattention-overview.png)

- Standard attention writes the $N\times N$ score matrix to HBM, reads it back
  for the softmax, writes the probabilities, reads them again for the
  $\mathbf{P}\mathbf{V}$ product. The FLOPs are fine; the *traffic* is
  quadratic in $N$.
- FlashAttention never materializes the matrix: it tiles $\mathbf{Q}$,
  $\mathbf{K}$, $\mathbf{V}$ into SRAM-sized blocks and uses the **online
  softmax** (keep a running max and running sum, rescale as new blocks arrive).
  Exact attention, with $O(N^2 d^2 / M)$ HBM accesses for SRAM size $M$.
- In the backward pass it *recomputes* the scores from the blocks instead of
  storing them; more FLOPs, less traffic, faster. The $5as^2b$ activation term
  disappears.
- Reported: 3× faster GPT-2 training at sequence length 1K, memory linear in
  $N$ ([Dao et al. 2022](https://arxiv.org/abs/2205.14135)). FlashAttention-2
  and 3 restructure the same algorithm for newer hardware
  ([Dao 2023](https://arxiv.org/abs/2307.08691),
  [Shah et al. 2024](https://arxiv.org/abs/2407.08608)).


## Fused kernels in general

- Every elementwise op (add the residual, GeLU, dropout, scale) is at 0.2
  FLOP/byte: pure memory traffic. Chained unfused, each one reads and writes
  the whole activation tensor.
- A **fused kernel** does the chain in one pass through memory. The arithmetic
  is unchanged; the traffic drops by the number of ops fused.
- `torch.compile` fuses elementwise chains automatically; attention,
  normalization and the cross-entropy over a 150k vocabulary get hand-written
  kernels (FlashAttention, fused RMSNorm, chunked cross-entropy).
- Roofline reasoning tells you *which* kernels to fuse: the ones furthest left.


In the midterm project, `torch.compile` and `F.scaled_dot_product_attention`
(which dispatches to FlashAttention) are the first two things to turn on. Expect
a 1.5 to 3× speedup over the naive loop for free.


# Parallelism

Everything so far was one GPU. Past about 1B parameters (states) or past the
wall-clock you can tolerate, the job must be split. There are four ways to cut
it, they compose, and the right combination is determined by which link each
communication pattern has to cross.


## Four ways to split a training job

![The four axes of parallelism and what each GPU holds. Source: instructor figure.](figs/12-Training-at-Scale/parallelism-overview.png)

| Axis | Splits | Communicates | Volume per step |
|---|---|---|---|
| Data (DP) | the batch | gradients, once per step | $2N$ bf16 values |
| Tensor (TP) | each weight matrix | activations, every layer | $\propto s\,b\,h \times L$ |
| Pipeline (PP) | consecutive layers | activations, at stage boundaries | $\propto s\,b\,h$ |
| Expert (EP) | the MoE experts | tokens, to and from experts | $\propto s\,b\,h \times$ MoE layers |

## Data parallel and all-reduce

Each of $p$ GPUs holds a full replica, runs its own micro-batches, and the
gradients are averaged before the step:

- A ring all-reduce moves $2\,\frac{p-1}{p} \times (\text{gradient bytes})$
  through each GPU per step, independent of $p$ for large $p$
  ([Patarasuk and Yuan 2009](https://doi.org/10.1016/j.jpdc.2008.09.002)).
  For a 1B model in bf16: about 4 GB per GPU per step.
- Over NVLink (600 GB/s): 7 ms. Over 100 Gb/s Ethernet (12.5 GB/s): 0.3 s.
  A step on 16k tokens of a 1B model is about $10^{14}$ FLOPs, 0.8 s on an
  A100, so the Ethernet version loses 40% unless the all-reduce is overlapped
  with the backward pass (which frameworks do, layer by layer).
- DP is the simplest axis and the one you will use on the SCC with two to four
  GPUs: `DistributedDataParallel`. Its limit is memory: every GPU still holds
  all 16 bytes per parameter.


## ZeRO: shard the states, keep data parallelism

![Per-GPU memory for a 7.5B model on 64 GPUs under the three ZeRO stages ($K = 12$ bytes of optimizer state per parameter). Figure 1 of Rajbhandari et al., 2020.](figs/12-Training-at-Scale/zero-memory-stages.png)

- The insight: in data parallelism every GPU updates every parameter, which is
  $p$-fold redundant. Give each GPU a $1/p$ slice of the optimizer states
  (stage 1), of the gradients too (stage 2), and of the parameters (stage 3).
- Stages 1 and 2 cost no extra communication: a reduce-scatter of gradients
  plus an all-gather of updated parameters is the same $2N$ volume as an
  all-reduce. Stage 3 adds one all-gather of parameters before the backward:
  $3N$, 1.5× ([Rajbhandari et al. 2020](https://arxiv.org/abs/1910.02054), section 7).
- Memory per GPU falls from $16N$ to $16N/p$. With 8 GPUs a 7B model's states
  drop from 112 GB to 14 GB.


## FSDP: ZeRO-3 in PyTorch

```python
from torch.distributed.fsdp import fully_shard
# FSDP2 API, PyTorch >= 2.6 (earlier: torch.distributed._composable.fsdp)
for block in model.blocks:
    fully_shard(block)        # each block's params, grads, optimizer state sharded
fully_shard(model)
```

- Parameters live sharded. Before a block's forward, FSDP all-gathers its
  weights; after, it frees them. Before the backward, it all-gathers again;
  after, it reduce-scatters the gradients
  ([Zhao et al. 2023](https://arxiv.org/abs/2304.11277)).
- The all-gather for block $k+1$ overlaps with the compute of block $k$;
  the unit of sharding (one transformer block) is chosen so there is enough
  compute to hide the communication.
- Combine with mixed precision (shard the fp32 master, compute in bf16) and
  checkpointing. On a node with NVLink this trains models up to tens of
  billions of parameters with no other parallelism.


## Tensor parallel

![Megatron-style split of an MLP across two GPUs. Source: instructor figure after Figure 3 of Shoeybi et al., 2019.](figs/12-Training-at-Scale/tensor-parallel.png)

- Split the first MLP matrix $\mathbf{A}$ by **columns**: each GPU computes
  $\mathrm{GeLU}[\mathbf{X}\mathbf{A}_i]$ on its slice with no communication
  (GeLU is elementwise). Split the second matrix $\mathbf{B}$ by **rows**: each
  GPU produces a partial sum; one all-reduce gives $\mathbf{Z}$.
- Attention splits by heads the same way: Q, K, V projections by columns, the
  output projection by rows. Two all-reduces per layer forward, two backward
  ([Shoeybi et al. 2019](https://arxiv.org/abs/1909.08053)).
- Communication every layer, on activations: needs NVLink bandwidth. TP stays
  **inside a node**, almost always $\text{TP} \le 8$.
- Pays twice: memory for weights *and* activations both drop by the TP degree.


## Pipeline parallel

![GPipe schedule with four stages and four micro-batches. Source: instructor figure after Huang et al., 2019.](figs/12-Training-at-Scale/pipeline-bubble.png)

- Put layers $1$ to $L/p$ on GPU 0, the next block on GPU 1, and so on; only
  the activations at the boundary travel, once per micro-batch. Cheap enough to
  cross nodes over InfiniBand.
- The cost is the **bubble**: with $p$ stages and $m$ micro-batches the idle
  fraction is $(p-1)/(m+p-1)$; GPipe calls it negligible once $m \ge 4p$
  ([Huang et al. 2019](https://arxiv.org/abs/1811.06965)).
- 1F1B and interleaved schedules shrink the bubble and the activation memory
  held in flight ([Narayanan et al. 2021](https://arxiv.org/abs/2104.04473));
  DeepSeek-V3's DualPipe overlaps communication with it.
- Needs a large global batch to supply the micro-batches: another reason LLM
  batches are millions of tokens.


How many micro-batches it takes to make the bubble small:

In [ ]:
#| code-fold: true
#| fig-cap: "Pipeline bubble fraction against the number of micro-batches, for 2 to 16 stages."
#| fig-alt: "Log-x plot of idle fraction against micro-batches from 1 to 256 for 2, 4, 8 and 16 stages. Every curve starts near (p-1)/p and falls toward zero; deeper pipelines need more micro-batches to cross the dashed 10% line."
import numpy as np
import matplotlib.pyplot as plt

def bubble(p, m):
    return (p - 1) / (m + p - 1)

print(f"p = 4 stages, m = 4 micro-batches: bubble = {bubble(4, 4):.2f}")
for p in [2, 4, 8, 16]:
    m_10 = 9 * (p - 1)   # (p-1)/(m+p-1) <= 0.1  <=>  m >= 9(p-1)
    print(f"p = {p:>2}: bubble at m = 4p is {bubble(p, 4 * p):.2f}; under 10% needs m >= {m_10}")

m = np.arange(1, 257)
fig, ax = plt.subplots(figsize=(7, 3.5))
for p in [2, 4, 8, 16]:
    ax.plot(m, bubble(p, m), lw=2, label=f"p = {p}")
ax.axhline(0.1, color="gray", ls="--", lw=1)
ax.set(xscale="log", xlabel="micro-batches per step, m", ylabel="idle fraction (p-1)/(m+p-1)")
ax.legend(frameon=False)
plt.show()

## Expert parallel and context parallel

- **Expert parallel (MoE).** Each GPU holds a subset of experts. After the
  router picks the experts for each token, an **all-to-all** sends tokens to
  the GPUs holding those experts and a second all-to-all brings results back
  ([Lepikhin et al. 2020](https://arxiv.org/abs/2006.16668),
  [Fedus et al. 2022](https://arxiv.org/abs/2101.03961)). DeepSeek-V3 uses 64-way
  EP across 8 nodes and limits each token to at most 4 nodes to bound the
  traffic ([DeepSeek-AI 2024](https://arxiv.org/abs/2412.19437)).
- Load balance is the failure mode: if the router sends most tokens to one
  expert, one GPU does the work while the rest wait. That is why MoE training
  adds a balancing loss or bias (lecture 11).
- **Context (sequence) parallel.** For 128K-token contexts, the $s^2$
  attention and the $sbh$ activations do not fit even one layer on one GPU.
  Split the *sequence* across GPUs and pass K, V blocks around a ring
  ([Liu et al. 2023](https://arxiv.org/abs/2310.01889)). Llama 3 used CP = 16
  for its long-context stage.


## Communication costs: the links

| Link | Bandwidth | Where |
|---|---|---|
| NVLink, A100 / H100 | 600 / 900 GB/s | GPU to GPU inside a node |
| PCIe 4.0 ×16 | about 32 GB/s per direction | GPU to CPU, or GPUs without NVLink |
| InfiniBand NDR, one port | 400 Gb/s = 50 GB/s | node to node |
| 100 GbE | 12.5 GB/s | node to node, commodity |

Sources: NVIDIA datasheets, PCI-SIG and InfiniBand specifications.

- A 20 to 70× cliff between inside the node and outside it. That cliff decides
  the layout: the axis that talks most often (TP) stays inside; the ones that
  talk least (PP, DP) go across.
- Everything else is overlap: communication scheduled to hide behind compute.
  A layout is judged by how much of the step is exposed communication.
- On the SCC, GPUs in a node may or may not share NVLink; check
  `nvidia-smi topo -m` before assuming.


## Llama 3's 4D layout

![Sixteen GPUs under tensor, context, pipeline and data parallelism; the TP group is the innermost. Figure 5 of Grattafiori et al., 2024.](figs/12-Training-at-Scale/llama3-4d-parallelism.png)

| GPUs | TP | CP | PP | DP | Seq. len. | Tokens/batch | TFLOP/s per GPU | MFU |
|---|---|---|---|---|---|---|---|---|
| 8,192 | 8 | 1 | 16 | 64 | 8,192 | 16M | 430 | 43% |
| 16,384 | 8 | 1 | 16 | 128 | 8,192 | 16M | 400 | 41% |
| 16,384 | 8 | 16 | 16 | 8 | 131,072 | 16M | 380 | 38% |

Source: [Grattafiori et al. 2024](https://arxiv.org/abs/2407.21783), Table 4.

TP = 8 fills the node. PP = 16 spans 16 nodes. DP spans the rest. When the
context went to 128K, CP = 16 took the GPUs that DP had been using and the
global batch stayed at 16M tokens.

## A one-slide decision guide

1. **Does $16N$ bytes plus activations fit on one GPU?** Yes: data parallel
   (DDP), as many GPUs as you have. Turn on bf16, FlashAttention, `torch.compile`
   first; they are worth more than a second GPU.
2. **States do not fit, but one transformer block does?** FSDP / ZeRO-3 across
   the node. Add checkpointing before adding GPUs.
3. **One block's weights or activations do not fit, or FSDP's all-gathers are
   exposed?** Tensor parallel inside the node (TP $\le 8$), FSDP or DP across.
4. **More nodes than DP can use efficiently (the all-reduce over InfiniBand
   dominates)?** Pipeline parallel across nodes, with enough micro-batches to
   keep the bubble under 10%.
5. **Mixture of experts?** Expert parallel for the MoE layers, bounded by how
   many nodes a token may visit.
6. **Context over about 32K?** Context parallel for the attention.


Readable, current references for all of this:
[*The Ultra-Scale Playbook*](https://huggingface.co/spaces/nanotron/ultrascale-playbook)
(Hugging Face, 2025) and [*How to Scale Your Model*](https://jax-ml.github.io/scaling-book/)
(Google DeepMind, 2025).


# Scaling laws

The sections so far answer "how do I train a model of size $N$ on $D$ tokens
with the hardware I have". Scaling laws answer the prior question: for a given
compute budget $C = 6ND$, what $N$ and $D$ should I choose? This is where the
material from the SP26 LLM lecture's scaling-law slides now lives.


## Kaplan et al. 2020: loss is a power law in each resource

![Test loss against compute, dataset size and parameters, each on log-log axes, from runs on WebText2. Figure 1 of Kaplan et al., 2020.](figs/12-Training-at-Scale/kaplan-power-laws.png)

- With the other two resources unconstrained, $L(N) \propto N^{-0.076}$,
  $L(D) \propto D^{-0.095}$, $L(C_\text{min}) \propto C_\text{min}^{-0.050}$:
  straight lines over six orders of magnitude in $N$. Doubling the parameters
  lowers the loss by $2^{-0.076} \approx 5\%$.
- Shape (depth vs. width, number of heads) barely matters; scale does.
- Their compute-optimal allocation: $N_\text{opt} \propto C^{0.73}$, data only
  $\propto C^{0.27}$. "Spend the budget on a bigger model." GPT-3 followed it:
  175B parameters on 300B tokens, 1.7 tokens per parameter
  ([Kaplan et al. 2020](https://arxiv.org/abs/2001.08361),
  [Brown et al. 2020](https://arxiv.org/abs/2005.14165)).


## Hoffmann et al. 2022 (Chinchilla): the corrected allocation

![Iso-FLOP profiles: for each compute budget, loss against model size has a clear minimum; the minima give the compute-optimal $N$ and $D$, extrapolated to Gopher's budget. Figure 3 of Hoffmann et al., 2022.](figs/12-Training-at-Scale/chinchilla-isoflop.png)

- Over 400 models, 70M to 16B parameters, 5B to 500B tokens, each with its own
  tuned learning-rate schedule (Kaplan's runs had shared fixed schedules, which
  penalized the small models).
- Three independent estimates all give $N_\text{opt} \propto C^{a}$,
  $D_\text{opt} \propto C^{b}$ with $a \approx b \approx 0.5$: **scale
  parameters and tokens together**, about 20 tokens per parameter.
- The test: Chinchilla, 70B parameters on 1.4T tokens, same compute as Gopher
  (280B on 300B), beat it everywhere
  ([Hoffmann et al. 2022](https://arxiv.org/abs/2203.15556)). Most 2020–2022
  LLMs had been undertrained.


## The parametric fit

Their third approach fits all runs with

$$
\begin{aligned}
\hat L(N, D) &= E + \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}}, \\
E &= 1.69,\; A = 406.4,\; B = 410.7,\; \alpha = 0.34,\; \beta = 0.28.
\end{aligned}
$$

- $E$: the loss of an ideal model, the entropy of the text. Irreducible.
- $A/N^{\alpha}$: how much a perfectly trained $N$-parameter transformer falls
  short of ideal. Shrinks with parameters.
- $B/D^{\beta}$: how much a finite sample of $D$ tokens falls short of
  convergence. Shrinks with data.
- This additive form with an irreducible term is Hoffmann et al.'s; Kaplan's
  joint law $L(N,D) = [(N_c/N)^{\alpha_N/\alpha_D} + D_c/D]^{\alpha_D}$ has a
  different shape and no constant.
- Caveat: [Besiroglu et al. 2024](https://arxiv.org/abs/2404.10102) could not
  reproduce Hoffmann et al.'s fitted constants from the extracted data
  (implausibly narrow confidence intervals); their refit gives $a \approx 0.51$,
  in line with the other two approaches. The *form* is standard; treat the constants as
  approximate.


## Compute-optimal allocation: the derivation

Minimize $\hat L(N, D)$ subject to $C = 6ND$. Substitute $D = C/(6N)$:

$$
\hat L(N) = E + A\,N^{-\alpha} + B\,\Big(\tfrac{6}{C}\Big)^{\beta} N^{\beta}.
$$

Set $\mathrm{d}\hat L/\mathrm{d}N = 0$:
$$
-\alpha A\,N^{-\alpha-1} + \beta B\,\Big(\tfrac{6}{C}\Big)^{\beta} N^{\beta-1} = 0
\;\Rightarrow\;
N^{\alpha+\beta} = \frac{\alpha A}{\beta B}\Big(\frac{C}{6}\Big)^{\beta}.
$$


$$
N_\text{opt}(C) = G\Big(\frac{C}{6}\Big)^{\frac{\beta}{\alpha+\beta}},\quad
D_\text{opt}(C) = \frac{C}{6N_\text{opt}} = G^{-1}\Big(\frac{C}{6}\Big)^{\frac{\alpha}{\alpha+\beta}},\quad
G = \Big(\frac{\alpha A}{\beta B}\Big)^{\frac{1}{\alpha+\beta}}.
$$

With $\alpha = \beta$ both exponents are $\tfrac12$: $N_\text{opt}$ and
$D_\text{opt}$ grow as $\sqrt{C}$ and their ratio is a constant. With the
paper's $\alpha = 0.34$, $\beta = 0.28$: exponents $0.45$ and $0.55$.


## The fit, pictured

In [ ]:
#| code-fold: true
#| out-width: "80%"
#| fig-cap: "Left: iso-FLOP slices of $\\hat L(N, C/6N)$ with the paper's constants; the dots are the closed-form compute-optimal sizes. Right: $N_\\text{opt}(C)$ and $D_\\text{opt}(C)$. Computed from the published fit."
#| fig-alt: "Left: five U-shaped loss curves against parameters for budgets from 1e19 to 1e23 FLOPs, minima at roughly 200M, 600M, 2B, 5B and 15B parameters. Right: two straight lines on log-log axes, N_opt rising as C to the 0.45 and D_opt as C to the 0.55."
import numpy as np
import matplotlib.pyplot as plt

E, A, B, alpha, beta = 1.69, 406.4, 410.7, 0.34, 0.28   # Hoffmann et al. 2022, approach 3
budgets = [1e19, 1e20, 1e21, 1e22, 1e23]                # FLOPs; add your own

def loss(N, D):
    return E + A / N**alpha + B / D**beta

G = (alpha * A / (beta * B)) ** (1 / (alpha + beta))
def n_opt(C):
    return G * (C / 6) ** (beta / (alpha + beta))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.5, 3.6))
N = np.logspace(7.5, 12, 500)
for C, color in zip(budgets, plt.cm.Blues(np.linspace(0.35, 1, len(budgets)))):
    ax1.plot(N, loss(N, C / (6 * N)), color=color, lw=2, label=f"C = {C:.0e} FLOPs")
    ax1.plot(n_opt(C), loss(n_opt(C), C / (6 * n_opt(C))), "o", color=color, mec="white", ms=7)
ax1.set(xscale="log", ylim=(1.5, 3.6), xlabel="parameters N (tokens D = C / 6N)",
        ylabel="predicted loss", title="Iso-FLOP slices of the fitted loss")
ax1.legend(fontsize=8, loc="lower left", frameon=False)

C = np.logspace(18, 26, 200)
ax2.loglog(C, n_opt(C), lw=2, label=f"$N_{{opt}} \\propto C^{{{beta / (alpha + beta):.2f}}}$")
ax2.loglog(C, C / (6 * n_opt(C)), lw=2, label=f"$D_{{opt}} \\propto C^{{{alpha / (alpha + beta):.2f}}}$")
ax2.set(xlabel="compute budget C (FLOPs)", ylabel="parameters or tokens",
        title="Closed-form optimum from the same fit")
ax2.legend(frameon=False)
plt.tight_layout()
plt.show()

## Worked calculation: your compute budget

Use the empirical Chinchilla rule $D \approx 20N$, so $C = 6ND = 120N^2$.

1. One A100-day at 40% MFU: $C = 1.25\times10^{14} \times 86{,}400 = 1.1\times10^{19}$ FLOPs.
2. $N_\text{opt} = \sqrt{C/120} = \sqrt{9\times10^{16}} = 3\times10^{8}$: a **300M-parameter** model.
3. $D_\text{opt} = 20N = 6\times10^{9}$ tokens, about 20 GB of text ($\approx$3 to 4 bytes per token for English web text).


One A100-week: $N_\text{opt} \approx 800\text{M}$, $D \approx 16\text{B}$
tokens. These are the sizes at which the midterm project's experiments are
*informative*: a 1B model trained on 500M tokens is wasting most of its
parameters, and its loss says little about what the same recipe does at scale.


Quizzable: given a budget in GPU-hours and an MFU, find $N_\text{opt}$ and
$D_\text{opt}$; or given $N$, find how many tokens it takes to be
Chinchilla-optimal and how long that takes.


The same calculation for any budget, with the parametric fit's answer for
comparison. The fit puts the optimum at a somewhat higher token ratio at this
scale; with constants this uncertain, both say "a few hundred million
parameters on several billion tokens".

In [ ]:
import math

gpu_days, n_gpus = 1, 1
peak, mfu = 312e12, 0.40          # A100 bf16; use 989e12 for an H100
bytes_per_token = (3, 4)          # English web text, roughly

C = gpu_days * 86_400 * n_gpus * peak * mfu
N_opt = math.sqrt(C / 120)        # D = 20 N, so C = 6 N D = 120 N^2
D_opt = 20 * N_opt
print(f"C = {C:.2e} FLOPs")
print(f"rule of thumb D = 20N:  N_opt = {N_opt:.2e} parameters, D_opt = {D_opt:.2e} tokens"
      f" ({D_opt * bytes_per_token[0] / 1e9:.0f} to {D_opt * bytes_per_token[1] / 1e9:.0f} GB of text)")

E, A, B, alpha, beta = 1.69, 406.4, 410.7, 0.34, 0.28
G = (alpha * A / (beta * B)) ** (1 / (alpha + beta))
N_fit = G * (C / 6) ** (beta / (alpha + beta))
D_fit = C / (6 * N_fit)
print(f"parametric fit:         N_opt = {N_fit:.2e} parameters, D_opt = {D_fit:.2e} tokens"
      f" ({D_fit / N_fit:.0f} tokens per parameter)")

C_week = 7 * C
print(f"one A100-week (D = 20N): N_opt = {math.sqrt(C_week / 120):.1e}, D_opt = {20 * math.sqrt(C_week / 120):.1e} tokens")

## After Chinchilla: overtrained small models

![Training tokens per parameter for released models. Sources: the GPT-3, Gopher, Chinchilla, Llama 1, 2 and 3 papers and the Qwen3 technical report. Source: instructor figure.](figs/12-Training-at-Scale/tokens-per-param.png)

- Compute-optimal minimizes *training* cost for a target loss. A model is
  trained once and served billions of times; adding inference cost to the
  objective makes a smaller model trained for longer cheaper overall
  ([Sardana and Frankle 2024](https://arxiv.org/abs/2401.00448)).
- Hence Llama 3 8B on 15.6T tokens, about 1,900 tokens per parameter, and all
  Qwen3 models on 36T ([Yang et al. 2025](https://arxiv.org/abs/2505.09388)).
  Loss keeps improving past the optimum, just more slowly per FLOP
  ([Gadre et al. 2024](https://arxiv.org/abs/2403.08540)).
- Llama 3's own fit put the optimum at $3.8\times10^{25}$ FLOPs at 402B
  parameters and 16.55T tokens, and found the iso-FLOP curves flat near the
  minimum, so the flagship *is* roughly Chinchilla-optimal and the small models
  are deliberately overtrained
  ([Grattafiori et al. 2024](https://arxiv.org/abs/2407.21783), section 3.2.1).
- A separate axis, inference-time compute (more tokens of reasoning per
  answer), is lecture 21's subject ([Snell et al. 2024](https://arxiv.org/abs/2408.03314)).


# Data at scale

The scaling laws take $D$ as a number. In practice $D$ is 15 to 36 trillion
tokens that somebody had to collect, clean and weight, and the quality of that
work moves the loss as much as a generation of hardware. This section follows
the FineWeb and Llama 3 reports, which are the two most complete public
descriptions.


## Where 15T tokens come from

![The FineWeb processing pipeline. Figure from Penedo et al., 2024.](figs/12-Training-at-Scale/fineweb-pipeline.png)

- Start: 96 Common Crawl snapshots. Extract text from the raw HTML (WARC)
  with `trafilatura` rather than using Common Crawl's own text (WET), which
  keeps menus and boilerplate.
- Base filtering: URL blocklist, English language ID with fastText, the
  Gopher repetition and quality heuristics
  ([Rae et al. 2021](https://arxiv.org/abs/2112.11446)). About 36T tokens remain.
- Deduplicate, apply C4-style and custom heuristic filters, anonymize PII.
  Result: 15T tokens, and a model trained on it beats models trained on every
  other public web dataset ([Penedo et al. 2024](https://arxiv.org/abs/2406.17557)).


## Deduplication

![Per-snapshot MinHash deduplication beats global deduplication of all 96 snapshots. Figure from Penedo et al., 2024.](figs/12-Training-at-Scale/fineweb-dedup-per-snapshot.png)

- Why: duplicated text is memorized, wastes compute, and leaks evaluation sets.
  Removing near-duplicates lets a model reach the same loss in fewer steps
  ([Lee et al. 2022](https://arxiv.org/abs/2107.06499)).
- How: **MinHash** on 5-gram shingles, 112 hashes in 14 bands of 8; documents
  sharing a band are near-duplicates (about 75% similar), clustered, one kept.
  Scales to trillions of tokens on CPU nodes.
- The surprise: deduplicating *globally* across all snapshots removed so much
  that the remainder was worse than the original. Deduplicating each snapshot
  on its own was better. Llama 3 went the other way, global MinHash plus
  URL-level and line-level dedup (a line seen more than 6 times per 30M
  documents is removed). The right amount of dedup is empirical.


## Quality filtering

- **Heuristics** (cheap, run on everything): document length, fraction of
  alphabetic characters, repeated n-grams, symbol ratios, lines ending in
  punctuation, "lorem ipsum", dirty-word lists. Each one is ablated: FineWeb
  reports which C4 rules help; the terminal-punctuation rule gives the biggest
  individual boost but removes about 30% of tokens, so FineWeb drops it for the
  data loss, not the score.
- **Model-based classifiers** (the 2024 step change): train a small fastText
  or BERT-sized classifier to score documents, keep the top fraction.
  FineWeb-Edu scored 460k pages for "educational value" with
  Llama-3-70B-Instruct, trained a classifier on the scores, and kept 1.3T of
  15T tokens; the result jumps on MMLU and ARC
  ([Penedo et al. 2024](https://arxiv.org/abs/2406.17557)).
  DCLM found the same with a fastText classifier
  ([Li et al. 2024](https://arxiv.org/abs/2406.11794)). Llama 3 uses both
  fastText and Roberta-based classifiers trained on Llama 2 judgments.
- Every filter is a bet about what the benchmarks reward. "Educational" text
  helps knowledge benchmarks; it also tilts the distribution.


## Synthetic data

- **Textbooks, problems, code with tests, generated by a stronger model.**
  Phi-4 (14B) makes synthetic data the centre of its pretraining recipe and
  outperforms its GPT-4 teacher on STEM question answering
  ([Abdin et al. 2024](https://arxiv.org/abs/2412.08905)). Qwen3 and Llama 3
  both include synthetic math and code in pretraining.
- Why it works: you can target the distribution (reasoning chains, rare
  languages, long documents) and verify the answers (run the code, check the
  math) in a way web text never allows.
- The risk: **model collapse**. Train recursively on generated text and the
  tails of the distribution vanish, generation by generation
  ([Shumailov et al. 2024](https://arxiv.org/abs/2305.17493)). Mitigations are
  the obvious ones: keep real data in the mix, verify what you generate, and
  diversify the generators.


## Data mixing: one slide

- The corpus is many sources (web, code, math, books, multilingual) and the
  sampling proportions are a hyperparameter. Llama 3's final mix: about 50%
  general knowledge, 25% math and reasoning, 17% code, 8% multilingual
  ([Grattafiori et al. 2024](https://arxiv.org/abs/2407.21783), section 3.1.3).
- How that was chosen: train small models on candidate mixes, fit scaling laws
  per mix, pick the mix whose extrapolation wins. Cheaper alternatives learn
  the weights directly: DoReMi trains a small proxy model to reweight domains
  ([Xie et al. 2023](https://arxiv.org/abs/2305.10429)).
- **Annealing.** In the last few percent of training, as the learning rate
  decays to zero, upsample the highest-quality data. Llama 3 reports
  +24.0% on GSM8K and +6.4% on MATH for the 8B model from annealing alone, and
  uses a 40B-token anneal of a half-trained model as a fast test of whether a
  new dataset is worth adding.
- Curriculum is now standard: Qwen3 trains in three stages (general, then
  reasoning-heavy, then long-context) with different mixes.


# Instrumentation

A run that is set up correctly still has to be watched. Two signals cover most
of what goes wrong: the loss curve, and the throughput. Together with a clean
experiment log, they are what you will read every morning during the midterm
project.


## Reading a loss curve

![Schematic of a healthy pretraining run with one loss spike, over a warmup-plus-cosine learning-rate schedule. Source: instructor figure (synthetic curves).](figs/12-Training-at-Scale/loss-curves.png)

- **Shape.** Fast early drop, then a slow power-law decline that looks nearly
  flat on a linear axis. Plot loss against *log* tokens, or compare runs by
  the token count at which they reach a target loss.
- **Spikes.** Sudden jumps that may or may not recover. Causes: learning rate
  too high for the batch, a bad batch, precision overflow, attention logits
  growing without bound. Fixes in current recipes: QK-norm, z-loss, gradient
  clipping, lower peak LR; PaLM restarted from a checkpoint and skipped the
  offending batches ([Chowdhery et al. 2022](https://arxiv.org/abs/2204.02311)).
- **The anneal.** The loss drops noticeably as the LR goes to zero. Two runs
  compared mid-schedule are not comparable; compare at the end, or use a
  constant LR with a short cooldown per checkpoint.
- **Train vs. validation.** At one epoch over web-scale data they track each
  other; a gap means repeated data or contamination, not classical overfitting.


## Throughput and MFU: the number to watch

$$
\text{MFU} = \frac{\text{tokens/s} \times 6N}{\text{peak FLOP/s of the GPUs}}
$$

(add the attention term $12\,L\,H\,Q\,T$ per token for long contexts). Source:
[Chowdhery et al. 2022](https://arxiv.org/abs/2204.02311), appendix B.

| Run | MFU |
|---|---|
| Megatron-Turing NLG 530B, A100 (2022) | about 30% |
| PaLM 540B, TPU v4 (2022) | 46% |
| Llama 3 405B, H100 (2024) | 38 to 43% |

- MFU counts *useful* FLOPs; recomputation is not credited. Hardware FLOPs
  utilization (HFU) counts everything the chip did. MFU is the honest one.
- Below about 30% on one GPU something is wrong: data loading, a memory-bound
  kernel, a tiny micro-batch, Python overhead. Above 50% is excellent.
- Log tokens/s every step. It is the first thing to check after any change and
  the first thing that reveals a silent regression.


MFU from a measured throughput, and the throughput a target MFU implies:

In [ ]:
def mfu(tokens_per_s, N, peak_flops, n_gpus=1, n_layer=0, d_model=0, seq_len=0):
    flops_per_token = 6 * N + 12 * n_layer * d_model * seq_len   # 12 L H Q T, with H Q = d_model
    return tokens_per_s * flops_per_token / (n_gpus * peak_flops)

A100_BF16 = 312e12
N, L, d, T = 124e6, 12, 768, 1024        # a GPT-2-small-sized model
tokens_per_s = 150_000                   # what your training log reports

print(f"MFU, 6N only:           {mfu(tokens_per_s, N, A100_BF16):.1%}")
print(f"MFU, with attention:    {mfu(tokens_per_s, N, A100_BF16, n_layer=L, d_model=d, seq_len=T):.1%}")
print(f"tokens/s for 40% MFU:   {0.40 * A100_BF16 / (6 * N):,.0f}")

## Reading an experiment log

The `autoresearch` loop ([Karpathy 2026](https://github.com/karpathy/autoresearch)):
an agent may edit only `train.py`; every experiment trains for exactly five
minutes and is scored on validation **bits per byte**,

$$
\text{bpb} = \frac{L_\text{val}\,[\text{nats/token}]}{\ln 2} \times \frac{\text{tokens}}{\text{bytes}},
$$

which does not change when the tokenizer does. About 12 experiments an hour.

| id | change | val bpb | keep? |
|---|---|---|---|
| 017 | baseline, re-run | 1.312 | |
| 018 | lr 3e-3 → 6e-3 | 1.298 | yes |
| 019 | batch 32 → 64 (same lr) | 1.304 | yes |
| 020 | SwiGLU MLP | 1.301 | yes |
| 021 | depth 8 → 12 | 1.331 | no |

- **Noise floor first.** Two baselines, different seeds: if they differ by
  0.01 bpb, a 0.008 "win" is nothing. Fixed-time runs add another source:
  a slower model sees fewer tokens.
- **Confounds.** 018 and 019 changed the effective learning rate twice; 021 lost
  because a deeper model trains on fewer tokens in five minutes, not because
  depth is bad.
- **The brief.** Did the change violate the rules (touch the data, the
  evaluation, the time limit)? Agents optimize the number they are given.


## This lecture and the midterm project

- You will train a small decoder-only transformer on one SCC GPU, then run an
  autoresearch-style loop against it: the agent proposes changes, the harness
  trains and scores them, you read the log.
- The *science* is yours: pick the three changes that mattered, re-run each
  with several seeds, and explain *why* each helped, using this lecture's
  vocabulary: was it compute-bound or memory-bound, did it change tokens seen
  per minute, did it move the model toward or away from Chinchilla-optimal?
- Sizing comes from section 5: with a few A100-hours per experiment you are in
  the 50M to 300M parameter range; make the data budget match.
- Speed comes from sections 2 and 3: bf16, `scaled_dot_product_attention`,
  `torch.compile`, the largest micro-batch that fits, a vocabulary padded to
  a multiple of 64. Measure MFU before and after.
- The in-class activity this week gives each team a real overnight log and
  asks which improvements survive a variance check.


# Summary

One line per idea; the recap deck uses these.


## Summary

- Training compute is $C \approx 6ND$: $2N$ forward, $4N$ backward, per token; the optimizer is negligible.
- A kernel's speed is $\min(\text{peak},\, I \times \text{bandwidth})$; below the ridge point (about 150 to 300 FLOP/byte) it is memory-bound.
- A bf16 GEMM's intensity is about its token count, so matmuls need a few hundred tokens each; elementwise ops are always memory-bound, so fuse them.
- Adam with mixed precision costs 16 bytes per parameter; activations cost $sbh(34 + 5as/h)$ per layer and depend on the micro-batch.
- bf16 keeps fp32's range and needs no loss scaling; fp16 and fp8 need scaling; reductions, norms and softmax stay in high precision.
- Gradient accumulation decouples micro-batch from global batch; checkpointing trades one extra forward for $O(\sqrt{L})$ activation memory.
- FlashAttention is exact attention that never writes the $N\times N$ matrix to HBM; I/O, not FLOPs, is what it saves.
- Data parallel replicates and all-reduces gradients; ZeRO/FSDP shards states so memory falls as $16N/p$; TP splits matrices inside a node; PP splits layers across nodes with a bubble of $(p-1)/(m+p-1)$; EP splits experts with an all-to-all.
- Scaling laws: loss is a power law in $N$, $D$, $C$; Chinchilla's $\hat L = E + A/N^\alpha + B/D^\beta$ with $C = 6ND$ gives $N_\text{opt} \propto C^{\beta/(\alpha+\beta)}$, about 20 tokens per parameter.
- Deployment cost favours overtrained small models: thousands of tokens per parameter for the 8B-and-under class.
- Data work (extraction, dedup, heuristic and classifier filtering, synthetic data, mixing, annealing) moves the loss as much as hardware does, and every filter is an ablation.
- Watch the loss curve on a log-token axis and tokens/s (MFU) every step; read experiment logs with a noise floor and an eye for confounds.

## Further reading

- [Hugging Face, *The Ultra-Scale Playbook: Training LLMs on GPU Clusters* (2025)](https://huggingface.co/spaces/nanotron/ultrascale-playbook): the most complete practical reference, with measured numbers.
- [Austin et al., *How to Scale Your Model* (Google DeepMind, 2025)](https://jax-ml.github.io/scaling-book/): roofline reasoning applied to every part of a transformer.
- [He, *Making Deep Learning Go Brrrr From First Principles* (2022)](https://horace.io/brrr_intro.html): compute, memory and overhead in one short essay.
- [Anthony et al., *Transformer Math 101* (EleutherAI, 2023)](https://blog.eleuther.ai/transformer-math/): the $6ND$ and 16-bytes-per-parameter arithmetic with worked examples.
- [Grattafiori et al., *The Llama 3 Herd of Models* (2024)](https://arxiv.org/abs/2407.21783), sections 3 and 4: data, scaling laws, parallelism and reliability for one real run.
- [Penedo et al., *The FineWeb Datasets* (2024)](https://arxiv.org/abs/2406.17557): every data-curation decision ablated.